In [144]:
import numpy as np
import pandas as pd
import plotly.express as px

In [145]:
np.random.seed(0)

In [146]:
threshold_min, threshold_max, threshold_delta = 0., 1., 0.1

In [147]:
def bayesian_update(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [148]:
def manipulation_thresholds(thresholds, priors, c):
    if np.sum(priors) == 0:
        return thresholds
    thresholds_m = np.maximum(0, thresholds - (bayesian_update(priors) / c))

    if len(thresholds_m) > 1:
        priors_updated = []
        thresholds_updated = []
        thresholds_m_updated = []
        prev = -np.inf
        for i, threshold_m in enumerate(thresholds_m):
            if threshold_m <= prev:
                del priors_updated[i-1]
                del thresholds_updated[i-1]
                del thresholds_m_updated[i-1]

                priors_updated.append(priors[i-1] + priors[i])
            else:
                priors_updated.append(priors[i])
            thresholds_updated.append(thresholds[i])
            thresholds_m_updated.append(thresholds_m[i])
            prev = threshold_m
    else:
        priors_updated = priors
        thresholds_updated = thresholds
        thresholds_m_updated = thresholds_m

    return thresholds_m_updated, priors_updated, thresholds_updated

In [149]:
def balance_priors(priors, random=False):
    total = np.sum(priors)
    if total == 1:
        return priors
    indices = priors == 0
    remainder = 1 - total
    if random:
        p = np.random.rand(indices.sum())
        p = (p / p.sum()) * remainder
    else:
        p = remainder / indices.sum()
    priors[indices] = p
    return priors

In [150]:
def best_response(x_0, thresholds, manip_thresholds):
    best_responses = []
    for i in range(len(thresholds)):
        t = thresholds[i]
        m = manip_thresholds[i]
        if m <= x_0 < t:
            best_responses.append(t)
        else:
            best_responses.append(x_0)
    return best_responses

In [151]:
def accuracy_loss(BR, thresholds, threshold_true):
    losses = [0] * len(thresholds)
    for i, threshold in enumerate(thresholds):
        for br in BR:
            y_star = int(br[i] >= threshold_true)
            y_hat = int(br[i] >= threshold)
            losses[i] += np.abs(y_star-y_hat)
    return np.array(losses) / len(BR)

In [152]:
num_agents = 10000
X = np.random.uniform(size=num_agents)
thresholds = np.arange(threshold_min+threshold_delta, threshold_max, threshold_delta).round(4)

priors = np.zeros_like(thresholds)
# priors[1] = 0.78
# priors[7] = 0.15
priors = np.array([0.00980328, 0.78, 0.0127752, 0.01076697, 0.00973307, 0.00756761, 0.0115374 , 0.15, 0.00781648])

balance_priors(priors, random=True)
print(np.sum(priors))
# assert np.sum(priors) == 1

pd.DataFrame({"threshold": thresholds, "priors": priors}).round(3).T

1.00000001


,0,1,2,3,4,5,6,7,8
threshold,0.10,0.20,0.300,0.400,0.50,0.600,0.700,0.80,0.900
priors,0.01,0.78,0.013,0.011,0.01,0.008,0.012,0.15,0.008


In [153]:
c = 5
threshold_true = 0.5
n = len(thresholds)
# partitions = [[i] for i in range(n)]
partitions = [[0,1], [2,3], [4,5], [6,7,8]]

In [154]:
results = {
    "threshold": [],
    "prior": [],
    "partition": [],
    "accuracy_loss": [],
    "partition_loss": [],
    "manip_threshold": [],
}

for i, partition in enumerate(partitions):
    threshold_p = thresholds[partition]
    priors_p = priors[partition]

    manip_thresh_p, priors_p, threshold_p = manipulation_thresholds(threshold_p, priors_p, c)
    BR = []
    for x in X:
        br = best_response(x, threshold_p, manip_thresh_p)
        BR.append(br)

    acc_loss = accuracy_loss(BR, threshold_p, threshold_true)
    partition_loss = np.dot(acc_loss, bayesian_update(priors_p))

    for ti, t in enumerate(threshold_p):
        results["threshold"].append(t)
        results["prior"].append(priors_p[ti])
        results["partition"].append(f"{i}")
        results["accuracy_loss"].append(acc_loss[ti])
        results["partition_loss"].append(partition_loss)
        results["manip_threshold"].append(manip_thresh_p[ti])

In [155]:
pd.DataFrame(results).T

,0,1,2,3,4,5,6
threshold,0.2,0.3,0.4,0.5,0.6,0.8,0.9
prior,0.789803,0.012775,0.010767,0.009733,0.007568,0.161537,0.007816
partition,0,1,1,2,2,3,3
accuracy_loss,0.5027,0.3099,0.1917,0.0,0.0122,0.1226,0.385
partition_loss,0.5027,0.255841,0.255841,0.005336,0.005336,0.134711,0.134711
manip_threshold,0.002482,0.19147,0.30853,0.387483,0.512517,0.622856,0.890769


In [156]:
print(f"True threshold: {threshold_true}")
fig = px.scatter(results, x = "threshold", y="partition_loss", color="partition", hover_data=["accuracy_loss"])

fig.show()

True threshold: 0.5
